# IMEN266 · HW 4 workspace (Ch.7)

Statement: `hw4.pdf` (PLMS / repo). This notebook is your **computational
workspace** for Parts B–C and your **prompt log** for Part D.
Part A is pen-and-paper first — no cells here on purpose.

▶ Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch07/homework/HW4.ipynb`

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
try:
    from imen266.renewal import (simulate_renewal, count, renewal_function, sampler_from,
                                 samplers_same_mean, alternating_renewal, age_residual_samples)
except ImportError:                                  # Colab: fetch the course package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/youngmko/imen266-2026.git"], check=True)
    from imen266.renewal import (simulate_renewal, count, renewal_function, sampler_from,
                                 samplers_same_mean, alternating_renewal, age_residual_samples)
from imen266.ctmc import poisson_process
rng = np.random.default_rng()      # your results should NOT depend on a seed
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": .3})

---
## Part B2 — Correct numbers (formulas on paper, numbers here)

Bus headways $X\sim$ uniform$(10,30)$ min, passengers $\PP(3)$ per minute, 1 won per
passenger-minute. Fill $E[X^2]$ and the renewal-reward formulas by hand; the grid
solver gives $m(t)$ for the two times asked.

In [ ]:
# --- B2 skeleton -------------------------------------------------------------
lam = 3.0                                   # passengers per minute
EX  = 20.0                                  # mean headway (minutes)
EX2 = np.nan                                # TODO: E[X^2] for uniform(10, 30)
W   = np.nan                                # TODO: mean wait of a passenger (minutes) — the remaining-time formula
L   = np.nan                                # TODO: mean number waiting (passengers)
pay = np.nan                                # TODO: payment rate (won per minute) = E[passenger-minutes per cycle] / E[X]

if np.isnan([EX2, W, L, pay]).any():
    print("fill EX2, W, L, pay first")
else:
    print(f"E[X^2] = {EX2:.3f}   W = {W:.3f} min   L = {L:.3f}   payment {pay:.3f} won/min   (Little: lam W = {lam*W:.3f})")

# the renewal function of uniform(10, 30) headways on [0, 60]: m(15) is exact on paper, m(60) is numerical
grid, m = renewal_function(stats.uniform(10, 20), 60.0)
print("m(15) =", np.interp(15, grid, m).round(4), "  (on paper: P(X <= 15) = 0.25, since a second bus needs S_2 >= 20)")
print("m(60) =", np.interp(60, grid, m).round(4), "  asymptote t/mu + (sigma^2 - mu^2)/(2 mu^2) =", round(60/20 + (400/12 - 400)/(2*400), 4))
plt.figure(); plt.plot(grid, m, label="m(t), grid solver"); plt.plot(grid, grid/20, "k--", lw=1, label="t / E[X]")
plt.xlabel("t (min)"); plt.ylabel("m(t)"); plt.legend(); plt.title("Expected number of buses: not t/E[X] at finite t"); plt.show()

---
## Part B3 — Simulate the bus stop directly (no renewal formula inside)

Buses: cumulative sums of uniform$(10,30)$ gaps; passengers: a Poisson process;
each passenger waits for the next bus. $L$ is measured at random inspection
times; the payment is the total passenger-minutes divided by the horizon;
$m(15)$ and $m(60)$ are averages of bus counts over fresh replications.

In [ ]:
# --- B3 skeleton -------------------------------------------------------------
def simulate_bus_stop(T, rng, lam=3.0, a=10.0, b=30.0, n_inspect=20_000):
    """One run over [0, T] minutes: returns (W_hat, L_hat, payment_rate)."""
    buses = simulate_renewal(lambda r, n: r.uniform(a, b, n), T, rng, overshoot=True)
    pax = poisson_process(lam, T, rng); pax = pax[pax < buses[-1]]
    next_bus = buses[np.searchsorted(buses, pax, side="right")]
    waits = next_bus - pax                                            # one wait per passenger
    ts = rng.uniform(0, T, n_inspect)                                 # random inspection times for L
    k = np.searchsorted(buses, ts, side="right")
    last_bus = np.where(k > 0, buses[np.maximum(k - 1, 0)], 0.0)
    waiting = np.searchsorted(pax, ts, side="right") - np.searchsorted(pax, last_bus, side="right")
    return waits.mean(), waiting.mean(), waits[pax + waits <= T].sum() / T   # payment: passenger-minutes per minute

reps = 10
est = np.array([simulate_bus_stop(50_000, rng) for _ in range(reps)])
for name, col in zip(["W (min)", "L", "payment (won/min)"], est.T):
    print(f"{name:>18}: {col.mean():.3f} +/- {1.96*col.std(ddof=1)/math.sqrt(reps):.3f}   (95% CI over {reps} runs)")

# m(15), m(60): bus counts over fresh replications
counts = np.array([[count(simulate_renewal(lambda r, n: r.uniform(10, 30, n), 60.0, rng), t) for t in (15.0, 60.0)] for _ in range(20_000)])
print("m(15) ~", counts[:, 0].mean().round(4), "   m(60) ~", counts[:, 1].mean().round(4), "  (+/- about", (1.96*counts[:, 1].std()/math.sqrt(len(counts))).round(3), ")")
# TODO: compare with your B2 numbers and with the AI's Steps 1-5; which of its numbers were wrong, and by how much?

---
## Part C1 — How fast does $N(t)/t$ settle? (prediction first!)

Normal approximation: $N(t)\approx\mathcal N\big(t/\mu,\ \sigma^2t/\mu^3\big)$, so with
$\mu=1$ the relative error of $N(t)/t$ has standard deviation $\mathrm{cv}/\sqrt t$.

**C1 prediction (before running!):** $t_{5\%}$ for cv $=0, 0.5, 1, 2$ with a reason: ___

In [ ]:
# --- C1 skeleton: empirical t_5% ---------------------------------------------------------
def t_needed(sampler, tol=0.05, conf=0.95, reps=200, T=20_000):
    """Smallest grid time t such that |N(s)/s - 1| <= tol for all grid s >= t in a fraction >= conf of the replications."""
    ts = np.logspace(1, math.log10(T), 60)
    ok = np.zeros((reps, len(ts)), bool)
    for r in range(reps):
        S = simulate_renewal(sampler, T, rng)
        ok[r] = np.abs(count(S, ts)/ts - 1.0) <= tol
    good = np.array([ok[:, i:].all(axis=1).mean() for i in range(len(ts))])   # stays inside from t onward
    idx = np.flatnonzero(good >= conf)
    return ts[idx[0]] if len(idx) else np.inf

cases = {"deterministic (cv 0)": (0.0, lambda r, n: np.full(n, 1.0)),
         "Erlang(4) (cv 0.5)":   (0.5, lambda r, n: r.gamma(4, 0.25, n)),
         "exponential (cv 1)":   (1.0, lambda r, n: r.exponential(1.0, n)),
         "lognormal (cv 2)":     (2.0, lambda r, n: r.lognormal(-0.5*math.log(5), math.sqrt(math.log(5)), n))}
print(f"{'gaps':>22} {'cv':>4} {'t_5% empirical':>16} {'your prediction':>16}")
for name, (cv, f) in cases.items():
    t_emp = t_needed(f)
    t_pred = np.nan                                      # TODO: your prediction from the normal approximation, as a function of cv
    print(f"{name:>22} {cv:>4} {t_emp:>16.0f} {t_pred:>16}")
# TODO: explain the ratio between consecutive rows; what happens for cv = 0?

---
## Part C2 — Insensitivity, explained by cycles

(a) one machine: six shapes, one answer. (b) two machines sharing one repair
person: the HW3 simulator, now with deterministic repairs.

In [ ]:
# --- C2(a): one machine, any shapes --------------------------------------------------------
m_up, m_rep = 2.0, 1.0
shapes = {"exponential":  lambda mean: (lambda r, n: r.exponential(mean, n)),
          "deterministic": lambda mean: (lambda r, n: np.full(n, mean)),
          "Erlang(4)":     lambda mean: (lambda r, n: r.gamma(4, mean/4, n)),
          "lognormal(cv=1.5)": lambda mean: (lambda r, n: r.lognormal(math.log(mean) - 0.5*math.log(1 + 1.5**2), math.sqrt(math.log(1 + 1.5**2)), n))}
print(f"{'up':>18} {'repair':>18}   availability   (E[Z]/(E[Z]+E[Y]) = {m_up/(m_up+m_rep):.4f})")
for up_name, up in shapes.items():
    for rep_name, rep in shapes.items():
        _, _, frac = alternating_renewal(up(m_up), rep(m_rep), 100_000, rng)
        print(f"{up_name:>18} {rep_name:>18}   {frac:.4f}")
# TODO (one sentence): which theorem makes this table obvious?

In [ ]:
# --- C2(b): two machines, one repair person (event-driven simulator from HW3; FCFS) --------------------
def simulate_repair(n_machines, n_crews, T, up_sampler, repair_sampler, rng):
    """Time fractions of the number of UP machines (0..n) over [0, T]; up_sampler(rng)/repair_sampler(rng) give one duration."""
    n = n_machines; t = 0.0
    up = np.ones(n, bool)
    fail = np.array([up_sampler(rng) for _ in range(n)]); done = np.full(n, np.inf)
    waiting = []; free = n_crews; occ = np.zeros(n + 1)
    while t < T:
        t_next = min(fail.min(), done.min(), T)
        occ[int(up.sum())] += t_next - t
        t = t_next
        if t >= T:
            break
        if fail.min() <= done.min():
            i = int(fail.argmin()); up[i] = False; fail[i] = np.inf
            if free > 0:
                free -= 1; done[i] = t + repair_sampler(rng)
            else:
                waiting.append(i)
        else:
            i = int(done.argmin()); done[i] = np.inf; up[i] = True; fail[i] = t + up_sampler(rng)
            if waiting:
                j = waiting.pop(0); done[j] = t + repair_sampler(rng)
            else:
                free += 1
    return occ / occ.sum()

one = lambda sampler: (lambda r: sampler(r, 1)[0])
T = 100_000
for rep_name in ["exponential", "deterministic"]:
    occ = simulate_repair(2, 1, T, one(shapes["exponential"](m_up)), one(shapes[rep_name](m_rep)), rng)
    print(f"two machines, one repair person, exp up / {rep_name:>13} repair: time fractions of # up = {occ.round(3)}   (exp/exp CTMC: (0.2, 0.4, 0.4))")
# TODO: why is one machine's up/down sequence NOT an alternating renewal process here? Name a regeneration epoch of the
#       joint process when the up times are exponential. Summarise the general result (ask your AI) in <= 5 sentences.

**C2 write-up (≤5 sentences):** *(double-click to edit)*

> ...

---
## Part C3 (optional) — The transient of the renewal function

In [ ]:
# --- C3 skeleton -------------------------------------------------------------
dists = {"Erlang(4)": stats.gamma(4, scale=0.25), "exponential": stats.expon(scale=1.0),
         "lognormal (cv 2)": stats.lognorm(s=math.sqrt(math.log(5)), scale=math.exp(-0.5*math.log(5)))}
plt.figure()
for name, D in dists.items():
    g, m = renewal_function(D, 12.0)
    mu, var = D.mean(), D.var()
    plt.plot(g, m - g/mu, label=f"{name}: limit (sigma^2 - mu^2)/(2 mu^2) = {(var - mu**2)/(2*mu**2):.3f}")
# deterministic gaps: m(t) = floor(t), no grid solver needed
g = np.linspace(0, 12, 1201); plt.plot(g, np.floor(g) - g, "k", lw=1, label="deterministic: floor(t) - t (no limit)")
plt.xlabel("t"); plt.ylabel("m(t) - t/mu"); plt.legend(fontsize=8); plt.title("The transient of the renewal function"); plt.show()
# TODO: why does the deterministic curve not converge? (lattice gaps; Companion Proof 7)

---
## Part D — Prompt log (keep it whenever AI was used)

| # | Where I used AI | Prompt (verbatim or faithful summary) | What it returned | What I verified / corrected |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

**Reflection (2–3 sentences):** where was the AI most useful, and where was it
least trustworthy, on this homework?

> ...